In [1]:
import pandas as pd
df = pd.read_csv("amazon.csv")


In [3]:
df.isnull().sum()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage: 183.3 KB


<h1> Pregunta de negocio: ¿ El mayor precio de descuentO aporta a un mejor rating ? </h1>

## ¿Qué son estos datos?

Qué es: Productos listados en Amazon India con su precio original, precio con descuento, porcentaje de descuento, rating promedio, cantidad de calificaciones, descripción y una muestra de reseñas de usuarios.

De dónde viene: Dataset público "Amazon Sales Dataset" publicado en Kaggle por Karkavelraja J. 

Qué representa una fila: Lo verificamos con código a continuación.

## Pregunta de Negocio: 

“¿En qué productos NO deberíamos invertir en promoción, porque probablemente van a quedar mal evaluados?”

In [5]:
print("Filas:", df.shape[0])
print("Productos únicos:", df["product_id"].nunique())

Filas: 1465
Productos únicos: 1351


In [ ]:
repetidos = df[df.duplicated(subset="product_id", keep=False)]
print("Filas exactamente duplicadas:", df.duplicated().sum())
print("Productos que aparecen más de una vez:", repetidos["product_id"].nunique())

# Que columnas cambian entre filas de un mismo producto
cambios = (repetidos.groupby("product_id").nunique() > 1).sum()
cambios[cambios > 0]

Filas exactamente duplicadas: 0
Productos que aparecen más de una vez: 92


discounted_price        3
actual_price            1
discount_percentage     4
rating_count           31
about_product           4
user_id                10
user_name              10
review_id              10
review_title           10
review_content         18
img_link               77
product_link           92
dtype: int64

In [7]:
ejemplo = repetidos["product_id"].iloc[0]
df[df["product_id"] == ejemplo][["product_id", "rating", "rating_count", "product_link"]]

,product_id,rating,rating_count,product_link
0,B07JW9H4J1,4.2,"24,269",https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
369,B07JW9H4J1,4.2,"24,270",https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
614,B07JW9H4J1,4.2,"24,269",https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...


In [8]:
from datetime import datetime

qid = df["product_link"].str.extract(r"qid=(\d+)")[0].astype(int)
print("Desde:", datetime.fromtimestamp(qid.min()))
print("Hasta:", datetime.fromtimestamp(qid.max()))

Desde: 2023-01-05 02:15:48
Hasta: 2023-01-05 10:00:17


**Conclusión: una fila = un producto en un resultado de búsqueda de Amazon, con hasta 8 reseñas concatenadas.**

- Hay 1.465 filas pero solo 1.351 productos únicos: 92 productos aparecen 2 o 3 veces.
- No son copias exactas: el rating es idéntico, pero cambian el `product_link` (posición en distintas búsquedas/categorías) y a veces el `rating_count` en una o dos unidades. Esto indica que el scraping capturó el mismo producto desde distintas páginas de búsqueda, minutos u horas apart.
- Cada fila trae en promedio ~8 reseñas pegadas en una misma celda, separadas por comas (no es una fila por reseña).
- Todos los datos se extrajeron el 5 de enero de 2023, en una ventana de ~8 horas. Es una foto de un solo momento, no una serie de tiempo.

## (5 C's)

### Clean: ¿el formato de los datos sirve para analizarlos?

Las columnas numéricas vienen como texto: los precios incluyen el símbolo ₹ y comas, el descuento incluye %, y `rating_count` usa el formato numérico indio (ej. `1,07,687`). Además `rating` contiene un valor no numérico (`|`). Ninguna de estas columnas se puede usar en un modelo sin convertirlas.

In [9]:
cols_numericas = ["actual_price", "discounted_price", "discount_percentage", "rating", "rating_count"]

# Todas vienen como texto en vez de número
print(df[cols_numericas].dtypes)
print()

# Cuántas filas tienen cada problema
print("Precios con ₹:", df["actual_price"].str.contains("₹").sum())
print("Descuentos con %:", df["discount_percentage"].str.contains("%").sum())
print("rating_count con comas:", df["rating_count"].str.contains(",", na=False).sum())

no_numerico = pd.to_numeric(df["rating"], errors="coerce").isna()
print("Ratings no numéricos:", df.loc[no_numerico, "rating"].tolist())

actual_price           str
discounted_price       str
discount_percentage    str
rating                 str
rating_count           str
dtype: object

Precios con ₹: 1465
Descuentos con %: 1465
rating_count con comas: 1137
Ratings no numéricos: ['|']
